## Metrics aggregation
Load all metric runs from `results/new_object_placement/poster/*/metrics.json` and plot each metric across runs.

In [ ]:
import sys
from pathlib import Path
repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [ ]:
from pathlib import Path
import json
import math
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

%matplotlib inline
pd.set_option("display.max_columns", None)

plt.style.use("seaborn-v0_8-whitegrid")
mpl.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


In [ ]:
base = Path('../results/new_object_placement/poster')
run_dirs = sorted([p for p in base.iterdir() if p.is_dir()], key=lambda p: int(p.name))
run_dirs = run_dirs[:35]
records = []
for run in run_dirs:
    metrics_path = run / 'metrics.json'
    if not metrics_path.exists():
        print(f'Skipping {run} (no metrics.json)')
        continue
    with metrics_path.open() as f:
        data = json.load(f)
    data['run'] = run.name
    records.append(data)

if not records:
    raise RuntimeError('No metrics found.')

df = pd.DataFrame(records).set_index('run').sort_index(key=lambda idx: idx.astype(int))
df.index.name = 'run'
df


In [ ]:
metrics = df.columns.tolist()
palette = plt.get_cmap('tab10')

def pretty_label(name: str) -> str:
    label_map = {
        'fid': 'FID',
        'ssim': 'SSIM',
        'iou': 'IOU',
    }
    key = name.lower()
    return label_map.get(key, name.replace('_', ' ').title())

def safe_name(name: str) -> str:
    keep = ''.join(ch if ch.isalnum() or ch in ('-', '_') else '_' for ch in name)
    return keep.strip('_').lower() or 'metric'

for idx, metric in enumerate(metrics):
    series = df[metric].astype(float)
    color = palette(idx % palette.N)
    bins = min(12, max(5, len(series) // 3))

    fig, ax = plt.subplots(figsize=(6, 3.6), constrained_layout=True)
    ax.hist(series.values, bins=bins, color=color, edgecolor='white', alpha=0.85)

    ax.set_title(f"{pretty_label(metric)} Distribution")
    ax.set_xlabel(pretty_label(metric))
    ax.set_ylabel('Count')
    ax.grid(True, axis='y', linestyle='--', alpha=0.6)
    ax.set_axisbelow(True)

    filename = base / f"{safe_name(metric)}_hist.svg"
    fig.savefig(filename, dpi=300, bbox_inches='tight')
    print(f'Saved figure to {filename}')
    plt.show()
    plt.close(fig)
